# N26 — Tire Agent

This notebook builds the **Tire Agent**, the second sub-agent in the F1 multi-agent system.
Its job is to answer one question per stint: **how much tyre life is left before the degradation cliff?**

The agent wraps two ML artefacts from N09/N10:
- **TireDegTCN** — causal TCN trained on 2023–2025 tyre stints, one per compound (C1–C5 fine-tuned, C6 global)
- **MC Dropout** × 50 forward passes for P10/P50/P90 uncertainty quantification

Pipeline:

    stint_state (compound, tyre_life, recent laps)
           │
           ▼
    predict_tire_deg_tool ──► TireDegTCN single pass ──► deg_rate (s/lap)
    estimate_laps_to_cliff_tool ──► MC Dropout × 50  ──► P10 / P50 / P90
           │
           ▼
      tire_react_agent (LangGraph ReAct)
           │
           ▼
      TireOutput(deg_rate, laps_to_cliff_p10/p50/p90, warning_level)

Output consumed by N31 Orchestrator before calling the Pit Strategy Agent (N28).

Models loaded from `data/models/tire_degradation/`:

| File | Contents |
|---|---|
| `routing_config.json` | compound → bundle filename + window size |
| `tiredeg_C{1-5}_ft.pt` | fine-tuned bundle: state dict + scaler + hparams |
| `tiredeg_modelA_v4.pt` | global fallback (C6 / unknown) |
| `mc_dropout_calibration.json` | per-compound uncertainty sigma (s) |


---

Imports and `repo_root` walker — no notebook-specific logic.

## Step 0 — Setup & model loading

Load the per-compound TireDegTCN bundles exported by N10. Each `.pt` file is
self-contained: state dict, fitted scaler, feature list, window size, and
architecture hyperparameters. The `load_bundle()` function reconstructs the
model from those artefacts without importing from the training notebook.


In [ ]:
import json
import sys
from dataclasses import dataclass, field
from pathlib import Path

import fastf1
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from langchain_openai import ChatOpenAI            # noqa: E402
from langgraph.prebuilt import create_react_agent  # noqa: E402

Causal dilated conv block with LayerNorm + GELU — reproduced from [N10](../strategy/tire_degradation/N10_tiredeg_compound_finetuning.ipynb).

In [2]:
# TireDegTCN is redefined here because src/strategy/inference/tire_predictor.py
# contains the older EnhancedTCN (N09 architecture, BatchNorm + attention),
# which has a different state dict layout than the N10 fine-tuned bundles.

class CausalConv1dBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        self.pad  = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, padding=0)
        self.norm = nn.LayerNorm(out_ch)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = F.pad(x, (self.pad, 0))
        x = self.conv(x)
        return self.drop(F.gelu(self.norm(x.transpose(1, 2)).transpose(1, 2)))




Residual wrapper: two `CausalConv1dBlock`s with identity skip — from [N10](../strategy/tire_degradation/N10_tiredeg_compound_finetuning.ipynb).

In [3]:
class TCNResidualBlock(nn.Module):
    def __init__(self, ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        self.net  = nn.Sequential(
            CausalConv1dBlock(ch, ch, kernel_size, dilation, dropout),
            CausalConv1dBlock(ch, ch, kernel_size, dilation, dropout),
        )

    def forward(self, x):
        return F.relu(self.net(x) + x)

Full TCN: linear input projection → residual blocks (dilation 2^i) → scalar output — from [N10](../strategy/tire_degradation/N10_tiredeg_compound_finetuning.ipynb).

In [4]:
class TireDegTCN(nn.Module):
    def __init__(self, n_features, d_model=64, n_layers=4, kernel_size=3, dropout=0.1):
        super().__init__()
        self.input_proj  = nn.Linear(n_features, d_model)
        self.blocks      = nn.ModuleList([
            TCNResidualBlock(d_model, kernel_size, 2**i, dropout)
            for i in range(n_layers)
        ])
        self.output_head = nn.Linear(d_model, 1)

    def forward(self, x, mask=None):
        x = self.input_proj(x).transpose(1, 2)
        for block in self.blocks:
            x = block(x)
        return self.output_head(x.transpose(1, 2)[:, -1, :]).squeeze(-1)


`TireAgentConfig` dataclass: resolves model paths, loads routing and calibration JSONs, exposes `load_bundle` / `load_all_bundles`.

In [ ]:
@dataclass
class TireAgentConfig:
    """Runtime configuration for the Tire Agent.

    Resolves all model paths relative to the repo root and loads — once — the
    three JSON artefacts produced by N10: routing config (compound → bundle file
    + window), MC Dropout calibration (per-compound uncertainty sigma), and
    encoding maps (label encodings for Team, Compound, AbsoluteCompound extracted
    from the training parquet). Also loads the circuit cluster map from the k=4
    parquet produced by N05.

    cliff_pit_soon_laps and cliff_monitor_laps are strategy rules of thumb:
    within 3 laps of the cliff the pace loss is hard to recover via undercut;
    within 7 there is still time to plan but the stop should be on the radar.
    Both are exposed as dataclass fields so callers can override them.
    """

    n_mc: int = 50
    model_name: str = "local-model"
    cliff_pit_soon_laps: int = 3
    cliff_monitor_laps: int = 7

    def __post_init__(self):
        root = Path.cwd()
        while not (root / ".git").exists():
            root = root.parent

        self._model_dir = root / "data" / "models" / "tire_degradation"
        self.export_dir = root / "data" / "models" / "agents"
        self.export_dir.mkdir(parents=True, exist_ok=True)

        fastf1.Cache.enable_cache(str(root / "data" / "cache" / "fastf1"))

        with open(self._model_dir / "routing_config.json") as f:
            self.routing_cfg: dict = json.load(f)
        with open(self._model_dir / "mc_dropout_calibration.json") as f:
            self.mc_calibration: dict = json.load(f)
        with open(self._model_dir / "encoding_maps.json") as f:
            enc = json.load(f)

        self.team_id_map: dict           = enc["team_id"]
        self.compound_id_map: dict       = enc["compound_id"]
        self.abs_compound_id_map: dict   = enc["absolute_compound_id"]
        self.compound_hardness_map: dict = enc["compound_hardness"]

        _cluster_df = pd.read_parquet(
            root / "data" / "processed" / "circuit_clustering" / "circuit_clusters_k4.parquet"
        )
        self.circuit_cluster_map: dict = dict(
            zip(_cluster_df["GP_Name"], _cluster_df["Cluster"].astype(int))
        )

        self.mc_sigma_fallback = float(
            np.mean([v["mean_sigma_s"] for v in self.mc_calibration.values()])
        )

    def load_bundle(self, compound_id: str) -> dict:
        """Load a compound bundle and attach an instantiated TireDegTCN in eval mode.

        Each .pt file is a self-contained dict exported by N10 that stores the
        state dict, fitted StandardScaler, feature name list, window size, and
        architecture hyperparameters — enough to reconstruct and run the model
        without importing from the training notebook.
        """
        cfg    = self.routing_cfg[compound_id]
        bundle = torch.load(self._model_dir / cfg["bundle"], map_location="cpu", weights_only=False)
        model  = TireDegTCN(bundle["n_features"], **bundle["model_hparams"])
        model.load_state_dict(bundle["state_dict"])
        model.eval()
        bundle["model"] = model
        return bundle

    def load_all_bundles(self) -> dict:
        """Load every compound defined in routing_config; return {compound_id: bundle}."""
        return {cid: self.load_bundle(cid) for cid in self.routing_cfg}

Instantiate config, load all six compound bundles from `data/models/tire_degradation/`, and connect the LM Studio LLM.

In [6]:
CFG     = TireAgentConfig()
BUNDLES = CFG.load_all_bundles()

llm = ChatOpenAI(
    model=CFG.model_name,
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0,
)

Verify all bundles loaded correctly — window sizes, feature count, scaler type, and MC calibration coverage.

In [ ]:
print(f"Compounds loaded    : {list(BUNDLES.keys())}")
for cid, b in BUNDLES.items():
    print(f"  {cid}  window={b['window']:>2}  features={b['n_features']}  scaler={type(b['scaler']).__name__}")
print(f"MC calibration      : {list(CFG.mc_calibration.keys())}  fallback sigma={CFG.mc_sigma_fallback:.4f}s")
print(f"Teams in map        : {len(CFG.team_id_map)}")
print(f"Circuits in cluster : {len(CFG.circuit_cluster_map)}")
print(f"Cliff thresholds    : PIT_SOON < {CFG.cliff_pit_soon_laps} laps | MONITOR < {CFG.cliff_monitor_laps} laps")
print(f"EXPORT_DIR          : {CFG.export_dir}")

### Step 0 — Results

Six compound bundles loaded successfully (C1–C6). All are fine-tuned N10 variants
except C6 which falls back to the global `tiredeg_modelA_v4.pt`. Window sizes
range from 22 laps (C5) to 31 (C2), reflecting how many past laps each compound
variant needs to make a reliable prediction.

MC Dropout calibration is available for C2, C4, C5, C6. C1 and C3 use the
cross-compound mean sigma as fallback — uncertainty estimates for those compounds
will be slightly wider than the calibrated ones.


---

## Step 1 — `TireOutput` dataclass + feature preparation

Defines the agent's output structure and the full feature pipeline that converts a
FastF1 stint slice into the 42-feature scaled tensor the TCN expects.

Feature engineering replicates the N10 training pipeline exactly — same column names,
same rolling computations, same lagged values for leaky features (`DegradationRate`,
`DegAcceleration`), same scaler from the bundle. The pipeline is split into focused
helper functions (one per feature group) so each step is independently readable and testable.

`TireOutput` dataclass — structured agent output with `warning_level` derived from `laps_to_cliff_p10` using the thresholds in `CFG`.

In [ ]:
@dataclass
class TireOutput:
    """Structured output of the Tire Agent for one driver at one point in the race.

    The TCN produces a single scalar (predicted cumulative degradation) per forward
    pass. From that scalar and N_MC Monte Carlo passes we derive deg_rate and the
    P10/P50/P90 interval for laps remaining before the cliff. warning_level is
    derived automatically in __post_init__ so downstream agents (N28 Pit Strategy,
    N31 Orchestrator) get a categorical signal without re-implementing thresholds.
    """

    compound: str
    current_tyre_life: int
    deg_rate: float
    laps_to_cliff_p10: float
    laps_to_cliff_p50: float
    laps_to_cliff_p90: float
    warning_level: str = field(init=False)

    def __post_init__(self):
        if self.laps_to_cliff_p10 < CFG.cliff_pit_soon_laps:
            self.warning_level = "PIT_SOON"
        elif self.laps_to_cliff_p10 < CFG.cliff_monitor_laps:
            self.warning_level = "MONITOR"
        else:
            self.warning_level = "OK"

Timing and delta helpers — convert FastF1 timedeltas to seconds, compute rolling trends, and build lagged degradation features. `DegradationRate` and `DegAcceleration` are shifted by 1 lap to avoid the leakage flagged in the N10 feature manifest.

In [ ]:
def _add_timing_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Convert FastF1 Timedelta columns to float seconds.

    FastF1 returns LapTime and Sector times as pandas Timedelta objects.
    The TCN expects plain floats matching the N10 training format.
    LapsSincePitStop is aliased from TyreLife — both encode laps on the current set.
    """
    df["LapTime_s"]        = df["LapTime"].dt.total_seconds()
    df["Sector1_s"]        = df["Sector1Time"].dt.total_seconds()
    df["Sector2_s"]        = df["Sector2Time"].dt.total_seconds()
    df["Sector3_s"]        = df["Sector3Time"].dt.total_seconds()
    df["LapsSincePitStop"] = df["TyreLife"]
    return df


def _add_delta_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Compute lap time trend, degradation rate, and degradation acceleration.

    DegradationRate and DegAcceleration are flagged as leaky at the current
    timestep in tiredeg_feature_manifest.json. Using shift(1) replicates the
    training setup where the model sees only the previous lap's rates, not the
    current one being predicted.
    """
    delta                  = df["LapTime_s"].diff()
    df["LapTime_Delta"]    = delta.fillna(0)
    df["LapTime_Trend"]    = df["LapTime_s"].rolling(3, min_periods=1).mean()
    df["DegradationRate"]  = delta.shift(1).fillna(0)
    df["DegAcceleration"]  = df["DegradationRate"].diff().fillna(0)
    return df


def _add_prev_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Shift timing and speed measurements back one lap to create prev-lap context.

    The first lap of a stint has no predecessor — it is filled with the current
    lap's value (shift fill_value equivalent) to avoid introducing NaN into the
    scaler input.
    """
    for new_col, src_col in [
        ("Prev_LapTime",  "LapTime_s"),
        ("Prev_SpeedFL",  "SpeedFL"),
        ("Prev_SpeedI1",  "SpeedI1"),
        ("Prev_SpeedI2",  "SpeedI2"),
        ("Prev_SpeedST",  "SpeedST"),
        ("Prev_TyreLife", "TyreLife"),
    ]:
        df[new_col] = df[src_col].shift(1).fillna(df[src_col])
    return df


def _add_speed_delta_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Compute trap-speed deltas (current minus previous lap) for all four sensors."""
    for sensor in ("FL", "I1", "I2", "ST"):
        df[f"Speed{sensor}_Delta"] = df[f"Speed{sensor}"] - df[f"Prev_Speed{sensor}"]
    return df

Compound, fuel, and session helpers — load integer encodings from `CFG` (derived from `encoding_maps.json`), estimate fuel load with a linear model, and normalise lap times against session-level reference values.

In [ ]:
def _add_compound_cols(df: pd.DataFrame, compound_id: str) -> pd.DataFrame:
    """Set compound identity features from the label-encoding maps in CFG.

    All three encodings are constant within a stint — compound cannot change
    mid-stint. CompoundHardness is the inverse of AbsoluteCompoundID: C1=6
    (hardest), C6=1 (softest), as encoded in the N10 training data.
    """
    df["AbsoluteCompoundID"] = CFG.abs_compound_id_map.get(compound_id, 3)
    df["CompoundHardness"]   = CFG.compound_hardness_map.get(compound_id, 4)
    df["CompoundID"]         = CFG.compound_id_map.get(df["Compound"].iloc[0], 1)
    return df


def _add_fuel_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Estimate fuel load and its lap time penalty using a linear consumption model.

    Standard F1 fuel model: 110 kg at race start, ~1.6 kg consumed per lap.
    Each kilogram adds approximately 0.03 s to lap time (TUMFTM race simulation
    reference). These constants reproduce the fuel features computed in N07/N10.
    """
    df["FuelLoad"]   = (110.0 - df["LapNumber"] * 1.6).clip(lower=0)
    df["FuelEffect"] = df["FuelLoad"] * 0.03
    return df


def _add_session_cols(df: pd.DataFrame, session_meta: dict) -> pd.DataFrame:
    """Normalise lap times against session fastest lap and circuit cluster mean.

    lap_time_pct_of_race_fastest encodes how far off the pace a driver is in
    percentage terms. lap_time_vs_cluster_mean captures circuit-type context —
    high-speed tracks have different typical lap time distributions than street
    circuits. Both features were computed globally across all drivers in N10 training.
    """
    df["lap_time_pct_of_race_fastest"] = df["LapTime_s"] / session_meta["fastest_lap_s"] * 100
    df["lap_time_vs_cluster_mean"]     = df["LapTime_s"] - session_meta["cluster_mean_lap_s"]
    df["laps_remaining"]               = session_meta["total_laps"] - df["LapNumber"]
    df["mean_sector_speed"]            = (df["SpeedI1"] + df["SpeedI2"] + df["SpeedFL"] + df["SpeedST"]) / 4
    df["track_status_clean"]           = (df["TrackStatus"] == "1").astype(float)
    return df

`build_stint_features()`: orchestrates all helpers in N10 training order → returns a DataFrame with exactly 42 columns ready for the scaler.

In [ ]:
def build_stint_features(
    stint_laps: pd.DataFrame,
    compound_id: str,
    session_meta: dict,
) -> pd.DataFrame:
    """Compute all 42 TCN input features from a FastF1 stint slice.

    Orchestrates the feature helper functions in the same order they were applied
    during N10 training. The returned DataFrame has columns in bundle['feature_names']
    order so the pre-fitted StandardScaler in the bundle can be applied directly.

    Args:
        stint_laps: FastF1 laps for one driver and one stint, sorted by LapNumber
                    ascending. Required columns: LapTime, Sector1/2/3Time,
                    SpeedFL/I1/I2/ST, TyreLife, Position, Compound, LapNumber,
                    TrackStatus, Team, and weather cols (AirTemp, TrackTemp,
                    Humidity, Rainfall).
        compound_id: Pirelli compound ID string, e.g. 'C2'. Used to look up
                     integer encodings from CFG.
        session_meta: Dict with keys:
            fastest_lap_s (float) — session fastest clean lap time in seconds.
            cluster_mean_lap_s (float) — mean lap time for this circuit cluster.
            total_laps (int) — total scheduled race laps.
            cluster_id (int) — circuit cluster 0-3 from CFG.circuit_cluster_map.
            team_id (int) — driver team numeric ID from CFG.team_id_map.
            year (int) — season year.

    Returns:
        DataFrame with 42 float columns in bundle['feature_names'] order.
    """
    df = stint_laps.copy().reset_index(drop=True)

    df = _add_timing_cols(df)
    df = _add_compound_cols(df, compound_id)
    df = _add_fuel_cols(df)
    df = _add_delta_cols(df)
    df = _add_prev_cols(df)
    df = _add_speed_delta_cols(df)
    df = _add_session_cols(df, session_meta)

    df["Cluster"] = session_meta["cluster_id"]
    df["TeamID"]  = session_meta["team_id"]
    df["Year"]    = session_meta["year"]

    return df[BUNDLES[compound_id]["feature_names"]].astype(float)

`build_stint_tensor()`: applies the bundle's `StandardScaler`, pads short stints by repeating the first row on the left, and returns a `(1, window, 42)` tensor ready for the TCN forward pass.

In [ ]:
def build_stint_tensor(
    stint_laps: pd.DataFrame,
    compound_id: str,
    session_meta: dict,
) -> torch.Tensor:
    """Scale and tensorise a stint feature DataFrame for TCN inference.

    Applies the StandardScaler stored inside the compound bundle — fitted on
    the 2023-2024 training data in N10 — then pads or trims the sequence to
    the compound's window length. Short stints are left-padded by repeating
    the first row, assuming steady-state conditions before the stint started.

    Args:
        stint_laps: Raw FastF1 laps for one driver + stint, sorted ascending.
        compound_id: Pirelli compound ID, e.g. 'C2'.
        session_meta: Same dict passed to build_stint_features.

    Returns:
        Float32 tensor of shape (1, window, 42) on CPU.
    """
    bundle = BUNDLES[compound_id]
    window = bundle["window"]

    feat_df = build_stint_features(stint_laps, compound_id, session_meta)
    scaled  = bundle["scaler"].transform(feat_df.values)

    if len(scaled) >= window:
        seq = scaled[-window:]
    else:
        pad = np.tile(scaled[0], (window - len(scaled), 1))
        seq = np.vstack([pad, scaled])

    return torch.tensor(seq, dtype=torch.float32).unsqueeze(0)  # (1, window, 42)

In [ ]:
to = TireOutput(compound="C2", current_tyre_life=15, deg_rate=0.12,
                laps_to_cliff_p10=4.5, laps_to_cliff_p50=7.1, laps_to_cliff_p90=9.8)
print(to)
print()
print("warning_level logic  : PIT_SOON <", CFG.cliff_pit_soon_laps, "| MONITOR <", CFG.cliff_monitor_laps)
print("TireOutput fields    :", list(TireOutput.__dataclass_fields__))

### Step 1 — Results

`TireOutput` instantiates correctly and derives `warning_level` from `laps_to_cliff_p10`
against the configurable thresholds in `CFG`. With P10=4.5 the example lands in MONITOR
(4.5 ≥ 3, < 7).

Feature pipeline: 7 helper functions cover timing, compound encoding, fuel model,
delta/trend, prev-lap context, speed deltas, and session-relative normalisation.
`build_stint_features` orchestrates them and selects the 42 columns in the exact
order `bundle['feature_names']` expects. `build_stint_tensor` applies the N10-fitted
scaler and left-pads stints shorter than the compound's window.

---

---